In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from category_encoders.count import CountEncoder
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,average_precision_score,
    confusion_matrix, matthews_corrcoef,balanced_accuracy_score,cohen_kappa_score
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import (
    SVMSMOTE, BorderlineSMOTE,
    ADASYN, KMeansSMOTE, SMOTE
)
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.under_sampling import TomekLinks
from sklearn.svm import SVC
import os
from matplotlib.widgets import Lasso
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegressionCV, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.base import clone

import xgboost as xgb
import shap
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from imblearn.under_sampling import EditedNearestNeighbours

c:\Users\duyp6\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data

In [ ]:
df_train = pd.read_csv("data/final/processed_train_data.csv", index=False)

df_test = pd.read_csv("data/final/processed_test_data.csv", index=False)


In [3]:
columns_remove = ['SEQN', 'YearStart']

In [4]:
df_train.drop(columns=columns_remove, inplace=True)
df_test.drop(columns=columns_remove, inplace=True)

In [5]:
df_train = df_train[df_train['milk_consumption']<=5]
df_test = df_test[df_test['milk_consumption']<=5]

In [6]:
print(df_train['label'].value_counts())
print(df_test['label'].value_counts())

label
0.0    11831
1.0     6412
Name: count, dtype: int64
label
0.0    3091
1.0    1359
Name: count, dtype: int64


## Training:  SMOTE with SVMSMOTE

In [7]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18243 entries, 0 to 18242
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Gender                    18243 non-null  float64
 1   Age                       18243 non-null  float64
 2   Race                      18243 non-null  float64
 3   PIR                       18243 non-null  float64
 4   Weight                    18243 non-null  float64
 5   Height                    18243 non-null  float64
 6   BMI                       18243 non-null  float64
 7   WaistCircumference        18243 non-null  float64
 8   Hba1c                     18243 non-null  float64
 9   FastingGlucose            18243 non-null  float64
 10  Albumin                   18243 non-null  float64
 11  ALT                       18243 non-null  float64
 12  AST                       18243 non-null  float64
 13  AlkalinePhosphotase       18243 non-null  float64
 14  BUN   

In [8]:
category_columns = [
    'Gender','Race', 'milk_consumption', 'label'
]

unuseful_columns = [
    "ALT", "Weight", "Height",
    "FastingGlucose", "UricAcid", 
    "MeanPlateletVolume", "TotalCholesterol",
    "MeanCellVolumn", "MeanCellHemoglobin", 
    "RedCellDistributionWidth", "Creatinine"
]

In [10]:
df_train.drop(columns=unuseful_columns,inplace=True)
df_test = df_test[df_train.columns]

In [11]:
print(len(df_train.columns))
df_train.columns

21


Index(['Gender', 'Age', 'Race', 'PIR', 'BMI', 'WaistCircumference', 'Hba1c',
       'Albumin', 'AST', 'AlkalinePhosphotase', 'BUN', 'GGT', 'TotalBilirubin',
       'HDLCholesterol', 'Triglycerides', 'LDLCholesterol', 'Hemoglobin',
       'Hematocrit', 'PlateletCount', 'milk_consumption', 'label'],
      dtype='object')

In [52]:
#model definition
from sklearn.ensemble import AdaBoostClassifier

dt_setups = {
    "Shallow": DecisionTreeClassifier(criterion="gini", max_depth=5, min_samples_split=10, min_samples_leaf=4, random_state=42),
    "Balanced": DecisionTreeClassifier(criterion="entropy", max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42),
    "Deep": DecisionTreeClassifier(criterion="gini", max_depth=20, min_samples_split=2, min_samples_leaf=1, random_state=42),
    "Randomized": DecisionTreeClassifier(criterion="log_loss", splitter="random", max_depth=None, min_samples_split=20, min_samples_leaf=6, random_state=42),
    "WideFeatures": DecisionTreeClassifier(criterion="entropy", splitter="best", max_depth=30, min_samples_split=5, min_samples_leaf=2, max_features="sqrt", random_state=42)
}
gbc_setups = {
    "Balanced": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    "ShallowFast": GradientBoostingClassifier(n_estimators=80, learning_rate=0.2, max_depth=2, subsample=0.8, random_state=42),
    "Regularized": GradientBoostingClassifier(n_estimators=200, learning_rate=0.01, max_depth=4, min_samples_split=20, min_samples_leaf=5, random_state=42),
    "Deep": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.9, random_state=42),
    "RobustSubsample": GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=5, subsample=0.7, max_features="sqrt", random_state=42),
    "Conservative": GradientBoostingClassifier(n_estimators=500, learning_rate=0.01, max_depth=3, subsample=0.8, max_features="log2", random_state=42)
}

rf_setups = {
    "Balanced": RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=42
    ),
    "ShallowFast": RandomForestClassifier(
        n_estimators=50, max_depth=5, max_features="sqrt", random_state=42
    ),
    "Deep": RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        min_samples_leaf=2, random_state=42
    ),
    "Regularized": RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_split=10,
        min_samples_leaf=4, max_features=0.6, random_state=42
    ),
    "Conservative": RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_split=20,
        min_samples_leaf=10, max_features="log2", random_state=42
    ),
    "RobustSubsample": RandomForestClassifier(#no ok
        n_estimators=150, max_depth=15, min_samples_split=5,
        min_samples_leaf=3, bootstrap=True, max_features="sqrt",
        random_state=42
    ),
    "Lightweight": RandomForestClassifier(
        n_estimators=50, max_depth=8, max_features=0.5, random_state=42
    ),
    "Heavy": RandomForestClassifier(
        n_estimators=1000, max_depth=None, min_samples_split=2,
        min_samples_leaf=1, max_features=None, random_state=42, n_jobs=-1
    ),
}

base_learners = [
    ('lightgbm', LGBMClassifier(n_estimators=100, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
    ('xgboost', XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)),
    ('RandomForest', rf_setups['Regularized']),
    # ('dt', dt_setups['Shallow']),  
    ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42)),
    #('SVM', SVC(kernel='rbf', C=10, gamma='auto', class_weight='balanced', probability=True, random_state=42)),
    # ('adaboost',AdaBoostClassifier(n_estimators=300,learning_rate=0.9, random_state=42)),
    ('nb', GaussianNB(var_smoothing= 1e-10)),
]


# ====== 3) Meta-learner ======
meta_learner = LogisticRegression(
    max_iter=3000,
    solver='saga',         
    C=0.2,  
    random_state=42,
    class_weight='balanced'
)

# ====== 4) Stacking classifier ======
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    stack_method="predict_proba",  
    passthrough=False,
)

In [56]:
from sklearn.linear_model import LogisticRegression

classifiers = {
    'LightGBM': LGBMClassifier(
        n_estimators=120, max_depth=6, num_leaves=31, 
        learning_rate=0.05, random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=80, learning_rate=0.066, 
        random_state=42, verbosity=0
    ),
    'GradientBoosting_80': GradientBoostingClassifier(
        n_estimators=80, learning_rate=0.05, random_state=42
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=100, random_state=42
    ),
    'Naive Bayes': GaussianNB(var_smoothing=1e-10),

    'LogisticRegression': LogisticRegression(
        solver='lbfgs', max_iter=500, random_state=42
    ),
    'ElasticNetLR': LogisticRegression(
        penalty='elasticnet', solver='saga', l1_ratio=0.5, 
        C=1.0, max_iter=1000, random_state=42
    ),

    #'SVM': SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42),
    
    'Stacking': stacking_clf,
}

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, precision_recall_curve, auc,matthews_corrcoef,balanced_accuracy_score,cohen_kappa_score
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE
from sklearn.svm import SVC
import os

def find_best_threshold(y_true, y_proba, metric="f1_macro"):

    thresholds = np.linspace(0.01, 0.99, 1000) 
    best_thr = 0.5
    best_score = -1.0

    y_true = np.array(y_true)

    for thr in thresholds:
        y_pred_thr = (y_proba >= thr).astype(int)

        if metric == "f1":
            score = f1_score(y_true, y_pred_thr, average="binary", zero_division=0)
        elif metric == "f1_macro":
            score = f1_score(y_true, y_pred_thr, average="macro", zero_division=0)
        elif metric == "balanced_acc":
            score = balanced_accuracy_score(y_true, y_pred_thr)
        else:
            raise ValueError(f"Metric '{metric}' not supported for threshold tuning.")

        if score > best_score:
            best_score = score
            best_thr = thr
            
    return round(best_thr, 4), round(best_score, 4)

# ====== 0) Train/test split ======
X_train_raw, X_val, y_train, y_val = train_test_split(
    df_train.drop(columns='label'), 
    df_train['label'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df_train['label'] 
)

idx_majority = y_train[y_train == 0].index
idx_minority = y_train[y_train == 1].index


keep_ratio = 0.5
keep_count = int(len(idx_minority) * keep_ratio)


np.random.seed(42)
idx_minority_reduced = np.random.choice(idx_minority, size=keep_count, replace=False)

final_indices = np.concatenate([idx_majority, idx_minority_reduced])

X_train_raw = X_train_raw.loc[final_indices]
y_train = y_train.loc[final_indices]

X_train_raw = X_train_raw.sample(frac=1, random_state=42)
y_train = y_train.loc[X_train_raw.index]

X_test_raw=df_test.drop(columns=['label'])
y_test=df_test['label']

numeric_cols = [col for col in df_train.columns if col not in category_columns]
categorical_encode_cols = ["Race"]

# ====== 1) Preprocessor ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_encode_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)

# ====== 3) SVMSMOTE setup ======
svmsmote = SVMSMOTE(
    random_state=42,
    sampling_strategy=0.6,
    k_neighbors=5,
    m_neighbors=10,
    svm_estimator=SVC(kernel="rbf", gamma="scale", C=1.0, random_state=42)
)

experiment_cases = {
    "smote+preprocess": [
        ('preprocessor', preprocessor),
        ('smote', svmsmote),
        ('classifier', classifiers)  
    ],
    "nosmote+preprocess": [
        ('preprocessor', preprocessor),
        ('classifier', classifiers)
    ],
    "smote+nopreprocess": [
        ('smote', svmsmote),
        ('classifier', classifiers)
    ],
    "nosmote+nopreprocess": [
        ('classifier', classifiers)
    ]
}

# ====== 4) Wrapper for imblearn pipelines ======
class ImblearnWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, pipeline):
        self.pipeline = pipeline
    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self
    def predict(self, X):
        return self.pipeline.predict(X)
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)
    
# ====== 5) Train & evaluate ======
rows_detailed = []
rows_summary = []

data_versions = [
   "Full_data_process (100%) + SVMSMOTE"
]


ROOT_DIR = r"notebooks/experiment_results/"
FOLDERNAME = "test_experiment"
STORE_PATH = os.path.join(ROOT_DIR, FOLDERNAME)

os.makedirs(STORE_PATH, exist_ok=True)

print("Results will be stored in:", STORE_PATH)
for version in data_versions:
    for case_name, steps in experiment_cases.items():
        summary_row = {
            "Data versioning": f"{version} - {case_name}",
            "Completeness": "✔",
            "Consistency": "✔",
        }

        for name, clf in classifiers.items():
            print(f"\n🚀 Training {name} on {version} [{case_name}]...")

            steps_with_clf = [(step, obj) if step != 'classifier' else ('classifier', clf) 
                              for step, obj in steps]

            pipeline = ImbPipeline(steps=steps_with_clf)
            wrapped_model = ImblearnWrapper(pipeline)

            try:
                # Training
                start = time.time()
                wrapped_model.fit(X_train_raw, y_train)
                train_time = time.time() - start

                # Threadhold tuning on validation set
                y_val_proba1 = wrapped_model.predict_proba(X_val)[:, 1]
                best_thr, _ = find_best_threshold(
                    y_val, 
                    y_val_proba1, 
                    metric="f1_macro" 
                )

                # Testing
                start_test = time.time()
                y_pred = wrapped_model.predict(X_test_raw)
                y_proba = wrapped_model.predict_proba(X_test_raw)
                y_proba1 = y_proba[:, 1]
                test_time = time.time() - start_test

                y_pred = (y_proba1 >= best_thr).astype(int)

                # Global metrics
                acc = accuracy_score(y_test, y_pred)
                f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
                f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
                acc_balanced = balanced_accuracy_score(y_test, y_pred)
                mcc = matthews_corrcoef(y_test, y_pred)
                cohen_kappa = cohen_kappa_score(y_test, y_pred)

                if len(np.unique(y_test)) == 2:
                    auc_roc = roc_auc_score(y_test, y_proba[:, 1])
                    auc_pr = auc(*precision_recall_curve(y_test, y_proba[:, 1])[1::-1])
                else:
                    auc_roc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
                    auc_pr = np.nan

                # Per-class metrics
                cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
                for i, cls in enumerate([0, 1]):
                    TP = cm[i, i]
                    FN = cm[i, :].sum() - TP
                    FP = cm[:, i].sum() - TP
                    TN = cm.sum() - (TP + FN + FP)

                    PPV = TP/(TP+FP) if (TP+FP) > 0 else 0
                    NPV = TN/(TN+FN) if (TN+FN) > 0 else 0
                    SEN = TP/(TP+FN) if (TP+FN) > 0 else 0
                    SPE = TN/(TN+FP) if (TN+FP) > 0 else 0

                    rows_detailed.append({
                        "Data versioning": f"{version} - {case_name}",
                        "Model": name,
                        "Label": cls,
                        "Training time": round(train_time, 4) if cls == 0 else None,
                        "Test time": round(test_time, 4) if cls == 0 else None,
                        "ACC": round(acc, 4) if cls == 0 else None,
                        "ACC balanced": round(acc_balanced, 4) if cls == 0 else None,
                        "MCC": round(mcc, 4) if cls == 0 else None,
                        "CohenKappa": round(cohen_kappa, 4) if cls == 0 else None,
                        "F1_macro": round(f1_macro, 4) if cls == 0 else None,
                        "F1_weighted": round(f1_weighted, 4) if cls == 0 else None,
                        "ROC_AUC": round(auc_roc, 4) if cls == 0 else None,
                        "PR_AUC": round(auc_pr, 4) if cls == 0 else None,
                        "TP": TP, "FP": FP, "FN": FN, "TN": TN,
                        "PPV": round(PPV, 4), "NPV": round(NPV, 4),
                        "SEN": round(SEN, 4), "SPE": round(SPE, 4),
                    })

                    # Summary F1 per label
                    summary_row[f"F1_label{cls}_{name}"] = round(f1_score(y_test, y_pred, pos_label=cls), 3)

                summary_row["Training time(seconds)"] = round(train_time, 3)
                summary_row[name] = round(acc, 3)

                print(f" {name} [{case_name}] - ACC={acc:.4f}, F1_macro={f1_macro:.4f}, ROC_AUC={auc_roc:.4f}, PR_AUC={auc_pr:.4f}")

            except Exception as e:
                print(f" Error training {name} [{case_name}]: {e}")

        rows_summary.append(summary_row)

# ====== 6) Save detailed results only ======
detailed_df = pd.DataFrame(rows_detailed)

excel_path = os.path.join(STORE_PATH, "test_results.csv")
detailed_df.to_csv(excel_path, index=False)

print(f"\nExported detailed metrics to {excel_path}")

✅ Results will be stored in: D:\Study\KLTN\G8Vitamin\src\models\experiment\results\test_experiment

🚀 Training LightGBM on Full_data_process (100%) + SVMSMOTE [smote+preprocess]...
[LightGBM] [Info] Number of positive: 5679, number of negative: 9465
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000867 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5501
[LightGBM] [Info] Number of data points in the train set: 15144, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.375000 -> initscore=-0.510826
[LightGBM] [Info] Start training from score -0.510826
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fur

c:\Users\duyp6\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\duyp6\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\duyp6\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ LightGBM [smote+preprocess] - ACC=0.6753, F1_macro=0.6550, ROC_AUC=0.7476, PR_AUC=0.5672

🚀 Training XGBoost on Full_data_process (100%) + SVMSMOTE [smote+preprocess]...
✅ XGBoost [smote+preprocess] - ACC=0.6910, F1_macro=0.6649, ROC_AUC=0.7484, PR_AUC=0.5650

🚀 Training GradientBoosting_80 on Full_data_process (100%) + SVMSMOTE [smote+preprocess]...
✅ GradientBoosting_80 [smote+preprocess] - ACC=0.6836, F1_macro=0.6544, ROC_AUC=0.7384, PR_AUC=0.5534

🚀 Training RandomForest on Full_data_process (100%) + SVMSMOTE [smote+preprocess]...
✅ RandomForest [smote+preprocess] - ACC=0.6710, F1_macro=0.6479, ROC_AUC=0.7386, PR_AUC=0.5568

🚀 Training Naive Bayes on Full_data_process (100%) + SVMSMOTE [smote+preprocess]...
✅ Naive Bayes [smote+preprocess] - ACC=0.6782, F1_macro=0.6378, ROC_AUC=0.7028, PR_AUC=0.4798

🚀 Training LogisticRegression on Full_data_process (100%) + SVMSMOTE [smote+preprocess]...
✅ LogisticRegression [smote+preprocess] - ACC=0.6787, F1_macro=0.6531, ROC_AUC=0.7400, PR_A

c:\Users\duyp6\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\duyp6\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\duyp6\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

c:\Users\duyp6\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\duyp6\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


c:\Users\duyp6\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ Stacking [smote+preprocess] - ACC=0.6937, F1_macro=0.6647, ROC_AUC=0.7467, PR_AUC=0.5662

✅ Exported detailed metrics to D:\Study\KLTN\G8Vitamin\src\models\experiment\results\test_experiment\test_results.csv


c:\Users\duyp6\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\duyp6\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
